# 01 · Data exploration (v1)

A light look at the processed datasets. **All logic lives in `src/vrp/`**; this notebook only loads
`data/processed/` and calls package functions. Run `make data` first.

Outcome columns (`rv_fwd21_*`, `payoff_*`, `vol_spread_pts`, `spx_fwd21_ret_log`) use data **after** day t;
see `docs/data_dictionary.md`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from vrp.config import load_config
from vrp.pipeline import read_processed
from vrp import sampling

cfg = load_config(ROOT / "config.yaml")
panel = read_processed(cfg, "daily_panel")
monthly = read_processed(cfg, "monthly_sample")
events = pd.read_csv(cfg.path("processed") / "events.csv", parse_dates=["start_date", "end_date"])
print(f"daily panel: {len(panel):,} rows {panel.index.min().date()} → {panel.index.max().date()}")
print(f"monthly sample: {len(monthly)} rows {monthly['month'].iloc[0]} → {monthly['month'].iloc[-1]}")

## The monthly sample

In [ ]:
monthly[["month", "vix_close", "rv_fwd21_ann_vol_pct", "payoff_scaled", "spx_fwd21_ret_log",
         "is_selloff", "selloff_decile", "post_2003"]].head()

In [ ]:
monthly[["vix_close", "rv_fwd21_ann_vol_pct", "iv_ann_var", "rv_fwd21_ann_var",
         "payoff_var", "payoff_scaled", "vol_spread_pts"]].describe().T.round(4)

## Worst ten months for a variance seller (descriptive only; tests come in v2)

In [ ]:
monthly.nsmallest(10, "payoff_scaled")[["month", "vix_close", "rv_fwd21_ann_vol_pct",
                                          "payoff_scaled", "spx_fwd21_ret_log"]].round(3)

## Forward-window overlap between consecutive months

In [ ]:
ov = sampling.window_overlap(monthly, panel)
print(sampling.overlap_summary(ov))
ov["overlap_days"].value_counts().sort_index()

## Event statistics (computed by `vrp.events`)

In [ ]:
stats = pd.read_csv(cfg.path("reports") / "event_stats.csv")
stats[["name", "max_vix_close", "max_vix_date", "worst_spx_log_ret", "spx_drawdown",
       "min_payoff_scaled_overlap"]].round(3)

## Figures

The figures are produced by `make figures` (`vrp.plots`) and saved in `outputs/figures/`.

In [ ]:
from IPython.display import Image
Image(filename=str(cfg.path("figures") / "03_scatter_iv_vs_rv.png"), width=600)